# Resilience: Resuming, Outcomes, Stalls, and Plain Logs

Long jobs crash, hang, and run where nobody is watching. This notebook shows how psnek helps with each:

1. **Resuming** an interrupted loop by skipping the items whose output already exists.
2. **Outcomes** record whether a bar finished, failed, or was interrupted.
3. **Stall detection** notices when progress stops, relative to the loop's own pace.
4. **Plain status lines** replace live drawing in CI logs and for AI agents.

In [1]:
import io
import logging
import tempfile
import time
from pathlib import Path

import psnek

logger = logging.getLogger("demo")
workdir = Path(tempfile.mkdtemp(prefix="psnek-demo-"))

## 1. Resume After a Crash

To make a long loop resumable, save each item's result to its own file and skip the items whose file already exists. Rerunning the loop after a crash, a kill, or a reboot then redoes only the unfinished work, even if items were added or reordered in between.

Two details make this reliable:

- Write each result under a temporary name, then rename it into place, so a crash mid-write never leaves a half-written file that looks finished.
- Drop the finished items before the loop rather than skipping them inside it, so the bar's total, rate, and ETA cover only the work that is left.

Here, indexing crashes at shard 60.

In [2]:
shards = [f"shard-{index:03d}" for index in range(100)]
indexes = workdir / "indexes"
indexes.mkdir()


def index_path(shard: str) -> Path:
    """Return the file that holds the index of ``shard``."""
    return indexes / f"{shard}.idx"


def index_shard(shard: str, crash_at: str | None = None) -> None:
    """Pretend to index ``shard`` into its own file, failing on ``crash_at``."""
    time.sleep(0.03)
    if shard == crash_at:
        raise OSError(f"disk full while writing {shard}")
    temporary = index_path(shard).with_suffix(".tmp")
    temporary.write_text(f"index of {shard}\n", encoding="utf-8")
    temporary.replace(index_path(shard))


def index_pending(crash_at: str | None = None) -> psnek.Track[str]:
    """Index the shards that have no index file yet, and return their bar."""
    pending = [shard for shard in shards if not index_path(shard).exists()]
    with psnek.track(pending, "index shards") as bar:
        for shard in bar:
            index_shard(shard, crash_at)
    return bar


def count_indexed() -> int:
    """Return how many shards have an index file."""
    return sum(index_path(shard).exists() for shard in shards)


try:
    index_pending(crash_at="shard-060")
except OSError as error:
    logger.error("indexing crashed: %s", error)
print(f"shards indexed: {count_indexed()}")

x index shards ###############+__________   60.0%     60/100         0:02   28.3it/s  failed OSError

indexing crashed: disk full while writing shard-060


shards indexed: 60


Running it again finds the 60 shards that already have an index file, so the bar covers only the remaining 40.

In [3]:
bar = index_pending()
print(
    f"outcome={bar.outcome}, indexed this run: {bar.n}, "
    f"shards indexed: {count_indexed()}"
)

# index shards ##########################  100.0%      40/40         0:01   28.6it/s  done

outcome=done, indexed this run: 40, shards indexed: 100


## 2. Outcomes

Every finished bar has an `outcome`: `done`, `failed`, or `interrupted`. Each is drawn with its own glyph and word, not just a color.

In [4]:
keep = psnek.Options(leave=True)
with psnek.Bar(total=20, desc="completes", options=keep) as done_bar:
    for _ in range(20):
        time.sleep(0.01)
        done_bar.update()

try:
    with psnek.Bar(total=20, desc="raises", options=keep) as failed_bar:
        for step in range(20):
            time.sleep(0.01)
            if step == 12:
                raise ValueError("bad record")
            failed_bar.update()
except ValueError as error:
    logger.error("job failed: %s", error)

with psnek.Bar(total=20, desc="cancelled", options=keep) as cancelled_bar:
    for step in range(20):
        time.sleep(0.01)
        if step == 7:
            cancelled_bar.interrupt()
            break
        cancelled_bar.update()

print([bar.outcome for bar in (done_bar, failed_bar, cancelled_bar)])

# completes ##########################  100.0%      20/20         0:00   81.1it/s  done
x raises    ###############+__________   60.0%      12/20         0:00   75.5it/s  failed ValueError
! cancelled #########.________________   35.0%       7/20         0:00   74.6it/s  interrupted

job failed: bad record


['done', 'failed', 'interrupted']


## 3. Stalls and Plain Status Lines

Without a terminal or notebook, such as in CI or when run by an AI agent, psnek prints plain status lines instead: a start line, one every 10 seconds, one whenever the bar's health changes, and a final summary.

Writing the bar to a `StringIO` uses that log mode here. The loop below hangs for 6 seconds midway, so psnek reports the stall, then the recovery.

In [5]:
log = io.StringIO()
with psnek.Bar(total=100, desc="ingest", options=psnek.Options(stream=log)) as bar:
    for step in range(100):
        time.sleep(6 if step == 50 else 0.02)
        bar.update()
        if step == 50:
            bar.panel({"waited_s": 6, "cause": "slow upstream"})
print(log.getvalue())

ingest: 0.0% 0/100 0:00<--:-- --it/s
ingest: 50.0% 50/100 0:06<~0:06 stalled 5s 40.3it/s
ingest: 51.0% 51/100 0:07<~0:06 7.1it/s (recovered)
waited_s  6
cause     slow upstream
ingest: 100.0% 100/100 0:08 done 14.0it/s



## Running Notebooks Headless

When a tool such as papermill or nbclient executes a notebook, set `PSNEK_MODE=log` to get these plain lines in the cell output instead.